In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.imputation import load_time_indexed_csv, impute_loads_by_gap_categories_safe, apply_imputed_columns, missing_summary
from src.time_features import add_calendar_features, get_calendar_feature_columns
from src.lag_features import add_lag_features, add_rolling_features, add_trend_features

import src.dataset_builder as dataset_builder
import src.profile_features as profile_features

In [3]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\resampled\LF_data_droped_15m.csv"
SAVE_DIR = Path(r"C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min") # change path when resolution is different # change path when resolution is different

In [4]:
df = load_time_indexed_csv(INPUT_PATH)
print(df.shape)
print(df.columns)

(11232, 9)
Index(['BA_Soc', 'BU_TotActPwr_Academy', 'BA_TotActPwr_BESS_AC_Panel1',
       'BA_TotActPwr_BESS_AC_Panel2', 'BU_TotActPwr_SDB_EL_Substation',
       'BU_TotActPwr_Tech_Room', 'PV_WS_AirTemp', 'PV_WS_Radiation',
       'PV_WS_RelHum'],
      dtype='str')


In [5]:
ms = missing_summary(df)
print(ms[ms["missing_count"] > 0])

n_rows_with_missing = df.isnull().any(axis=1).sum()
print(f"Rows with any missing values: {n_rows_with_missing}")

                                missing_count  missing_pct
BU_TotActPwr_SDB_EL_Substation           2362        21.03
BA_Soc                                    842         7.50
PV_WS_AirTemp                             447         3.98
PV_WS_Radiation                           447         3.98
PV_WS_RelHum                              447         3.98
BU_TotActPwr_Tech_Room                    105         0.93
BU_TotActPwr_Academy                      104         0.93
BA_TotActPwr_BESS_AC_Panel1               103         0.92
BA_TotActPwr_BESS_AC_Panel2               103         0.92
Rows with any missing values: 3103


In [6]:
df = add_calendar_features(df, include_extended=True)
calendar_cols = get_calendar_feature_columns(include_extended=True)

In [7]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

support_cols = [
    "BA_Soc",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
]

# lags = [1, 2, 3, 4, 8, 12, 24, 48, 96, 192, 288, 672]
# rolling_windows = [4, 12, 24, 48, 96, 288]
# std_windows = [12, 24, 48, 96]
# trend_pairs = [(1, 4), (1, 12), (1, 96), (96, 192)]

In [8]:
# df = add_lag_features(df, target_cols, lags=lags)

# df = add_rolling_features(
#     df,
#     target_cols,
#     rolling_windows=rolling_windows,
#     add_std_windows=std_windows,
# )

# df = add_trend_features(
#     df,
#     target_cols,
#     trend_pairs=trend_pairs,
# )


In [9]:
# for target_col in target_cols:
#     df = profile_features.add_day_ahead_history_features(
#         df,
#         target_col=target_col,
#         freq_minutes=15,
#     )

In [10]:
datasets = dataset_builder.build_multiple_target_datasets(
    df,
    target_cols=target_cols,
    support_cols=support_cols,
    calendar_cols=calendar_cols,
    include_calendar=True,
    include_lags=True,
    include_rolls=True,
    include_trends=True,
    include_history=True,
    drop_startup=True,
    startup_rows=0,
    drop_target_nan=False,
)

In [11]:
for name, dfx in datasets.items():
    print(name, dfx.shape)

BU_TotActPwr_Academy (11232, 18)
BA_TotActPwr_BESS_AC_Panel1 (11232, 18)
BA_TotActPwr_BESS_AC_Panel2 (11232, 18)
BU_TotActPwr_Tech_Room (11232, 18)
BU_TotActPwr_SDB_EL_Substation (11232, 18)


In [12]:
saved_files = dataset_builder.save_target_datasets_parquet(datasets, SAVE_DIR)

for k, v in saved_files.items():
    print(k, v, v.exists())

BU_TotActPwr_Academy C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BU_TotActPwr_Academy.parquet True
BA_TotActPwr_BESS_AC_Panel1 C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BA_TotActPwr_BESS_AC_Panel1.parquet True
BA_TotActPwr_BESS_AC_Panel2 C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BA_TotActPwr_BESS_AC_Panel2.parquet True
BU_TotActPwr_Tech_Room C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BU_TotActPwr_Tech_Room.parquet True
BU_TotActPwr_SDB_EL_Substation C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BU_TotActPwr_SDB_EL_Substation.parquet True
